Dataset Description:

This dataset contains 64,000 customers who last purchased within twelve months. The customers were involved in an e-mail test.
* 1/3 were randomly chosen to receive an e-mail campaign featuring Mens merchandise.
* 1/3 were randomly chosen to receive an e-mail campaign featuring Womens merchandise.
* 1/3 were randomly chosen to not receive an e-mail campaign.
During a period of two weeks following the e-mail campaign, results were tracked. Your job is to tell the world if the Mens or Womens e-mail campaign was successful.


Historical customer attributes at your disposal include:
* Recency: Months since last purchase.
* History_Segment: Categorization of dollars spent in the past year.
* History: Actual dollar value spent in the past year.
* Mens: 1/0 indicator, 1 = customer purchased Mens merchandise in the past year.
* Womens: 1/0 indicator, 1 = customer purchased Womens merchandise in the past year.
* Zip_Code: Classifies zip code as Urban, Suburban, or Rural.
* Newbie: 1/0 indicator, 1 = New customer in the past twelve months.
* Channel: Describes the channels the customer purchased from in the past year.

Another variable describes the e-mail campaign the customer received:
* Segment
  * Mens E-Mail
  * Womens E-Mail
  * No E-Mail

Finally, we have a series of variables describing activity in the two weeks following delivery of the e-mail campaign:
* Visit: 1/0 indicator, 1 = Customer visited website in the following two weeks.
* Conversion: 1/0 indicator, 1 = Customer purchased merchandise in the following two weeks.
* Spend: Actual dollars spent in the following two weeks.


Source: https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html



## Downloading the Dataset

In [ ]:
# Set up Kaggle API credentials using Colab Secrets
import os
from google.colab import userdata

# Ensure the .kaggle directory exists
!mkdir -p ~/.kaggle

# Write the kaggle.json file
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

# Set appropriate permissions for the file
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API credentials configured.")

Kaggle API credentials configured.


In [ ]:
# Import the Kaggle API client
import kaggle

# Define the Kaggle dataset path (e.g., 'segmentation/email-campaign-data')
# You need to find the correct path on Kaggle for the email campaign dataset.
# For example, it might look like 'user/dataset-name'
KAGGLE_DATASET_PATH = 'bofulee/kevin-hillstrom-minethatdata-e-mailanalytics'

# Download the dataset
# The 'p' flag is to specify the path where to download the dataset.
# We will download it to the current content directory.
!kaggle datasets download -d {KAGGLE_DATASET_PATH} -p /content/

print(f"Dataset downloaded to /content/")

# If the dataset is a zip file, you might need to unzip it
import zipfile
with zipfile.ZipFile('/content/kevin-hillstrom-minethatdata-e-mailanalytics.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')
print("Dataset unzipped.")

Dataset URL: https://www.kaggle.com/datasets/bofulee/kevin-hillstrom-minethatdata-e-mailanalytics
License(s): apache-2.0
100% 497k/497k [00:00<00:00, 509kB/s]

Dataset downloaded to /content/
Dataset unzipped.


## Load the Dataset into a Pandas DataFrame

In [ ]:
import pandas as pd

# Define the path to the extracted CSV file
DATA_FILE_PATH = '/content/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv'

# Load the CSV file into a pandas DataFrame
df = pd.read_csv(DATA_FILE_PATH)

# Display the first few rows of the DataFrame and its information
print("First 5 rows of the dataset:")
print(df.head())

First 5 rows of the dataset:
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

         segment  visit  conversion  spend  
0  Womens E-Mail      0           0    0.0  
1      No E-Mail      0           0    0.0  
2  Womens E-Mail      0           0    0.0  
3    Mens E-Mail      0           0    0.0  
4  Womens E-Mail      0           0    0.0  


In [ ]:
df_v1 = df.copy()

## Exploratory Data Analysis

In [ ]:
print("\nDataFrame Info:")
df.info()


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  object 
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  object 
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  object 
 8   segment          64000 non-null  object 
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), object(4)
memory usage: 5.9+ MB


In [ ]:
print(df.dtypes)

recency              int64
history_segment     object
history            float64
mens                 int64
womens               int64
zip_code            object
newbie               int64
channel             object
segment             object
visit                int64
conversion           int64
spend              float64
dtype: object


In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
numerical_cols = [col for col in numerical_cols if col not in ['visit', 'conversion']]
target_cols = ['visit', 'conversion', 'spend']
categorical_cols = df.select_dtypes(include=['object']).columns

In [ ]:
for col in categorical_cols:
    print(f"\nUnique values in column '{col}':")
    print(df[col].unique())
    print(f"\nValue counts for column '{col}':")
    print(df[col].value_counts(normalize=True))


Unique values in column 'history_segment':
['2) $100 - $200' '3) $200 - $350' '5) $500 - $750' '1) $0 - $100'
 '6) $750 - $1,000' '4) $350 - $500' '7) $1,000 +']

Value counts for column 'history_segment':
history_segment
1) $0 - $100        0.358906
2) $100 - $200      0.222719
3) $200 - $350      0.192016
4) $350 - $500      0.100141
5) $500 - $750      0.076734
6) $750 - $1,000    0.029047
7) $1,000 +         0.020438
Name: proportion, dtype: float64

Unique values in column 'zip_code':
['Surburban' 'Rural' 'Urban']

Value counts for column 'zip_code':
zip_code
Surburban    0.449625
Urban        0.400953
Rural        0.149422
Name: proportion, dtype: float64

Unique values in column 'channel':
['Phone' 'Web' 'Multichannel']

Value counts for column 'channel':
channel
Web             0.440891
Phone           0.437828
Multichannel    0.121281
Name: proportion, dtype: float64

Unique values in column 'segment':
['Womens E-Mail' 'No E-Mail' 'Mens E-Mail']

Value counts for column 'segm

In [ ]:
for col in numerical_cols:
  if col != 'spend':  # we will describe spend with the other target_cols
    print(f"\nDescriptive statistics for column '{col}':")
    print(df[col].describe())


Descriptive statistics for column 'recency':
count    64000.000000
mean         5.763734
std          3.507592
min          1.000000
25%          2.000000
50%          6.000000
75%          9.000000
max         12.000000
Name: recency, dtype: float64

Descriptive statistics for column 'history':
count    64000.000000
mean       242.085656
std        256.158608
min         29.990000
25%         64.660000
50%        158.110000
75%        325.657500
max       3345.930000
Name: history, dtype: float64

Descriptive statistics for column 'mens':
count    64000.000000
mean         0.551031
std          0.497393
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: mens, dtype: float64

Descriptive statistics for column 'womens':
count    64000.000000
mean         0.549719
std          0.497526
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: womens, dtype: float64


In [ ]:
for col in target_cols:
    print(f"\nDescriptive statistics for column '{col}':")
    print(df[col].describe())
    if col != 'spend':
      print(f"\nValue counts for column '{col}':")
      print(df[col].value_counts(normalize=True))


Descriptive statistics for column 'visit':
count    64000.000000
mean         0.146781
std          0.353890
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: visit, dtype: float64

Value counts for column 'visit':
visit
0    0.853219
1    0.146781
Name: proportion, dtype: float64

Descriptive statistics for column 'conversion':
count    64000.000000
mean         0.009031
std          0.094604
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: conversion, dtype: float64

Value counts for column 'conversion':
conversion
0    0.990969
1    0.009031
Name: proportion, dtype: float64

Descriptive statistics for column 'spend':
count    64000.000000
mean         1.050908
std         15.036448
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        499.000000
Name: spend, dtype: float64


In [ ]:
print(f'Proportion of users that visited the site post-campaign: {df[df['visit'] > 0].shape[0] / df.shape[0]}')
print(f'Proportion of users that purchased something from the site post-campaign > 0: {df[df['conversion'] > 0].shape[0] / df.shape[0]}')

Proportion of users that visited the site post-campaign: 0.14678125
Proportion of users that purchased something from the site post-campaign > 0: 0.00903125


From this, we can see that (not accounting for the campaign or variables directly related to it)

> Recency: Of the number of months since the last purchase, the mean was approx. 5.76 and the std was approx. 3.51

> History: Of the amounts of money spent last year, the mean was approx. \$242.09 and std approx. \$256.16

> Gender of Merchandise Purchased in the Past Year: In the past year, approx. 55.10% of customers purchased Men's merchandise and 54.97% purchased Women's merchandise

> Approx. 50.23% of the customers were newbies (new in the past 12 months).

> History Segment: Approx. 35.89% of customers are in history segment 1 (\$0-\$100), 22.27% in history segment 2 (\$100-\$200), 19.20% in history segment 3 (\$200-\$350), 10.01% in history segment 4 (\$350-\$500), 7.67% in history segment 5 (\$500-\$750), 2.90% in history segment 6 (\$750-\$1000), and 2.04% in history segment 7 (\$1000+)

> Zip Code: Approx. 44.96% of customers are in Suburban zip codes, 40.10% in Urban zip codes, and 14.94% in Rural zip codes

> Channel: Approx. 44.09% of customers used the Web, 43.78% used the Phone, and 12.13% used multiple channels

> Segment: Approx. 33.42% of customers received the women's email campaign, 33.29% received the men's email campaign, and 33.29% received no email campaign

**Post-campaign, plenty of users visited the site (≈14.68%), but still very few users actually purchased merchandise from the site (≈0.90%)**

## Causal Inference: A/B Test Analysis

To determine if the e-mail campaigns had a statistically significant causal effect, we will perform A/B testing. We'll compare the 'Mens E-Mail' and 'Womens E-Mail' groups (treatment groups) against the 'No E-Mail' group (control group) for each of the outcome variables: `visit`, `conversion`, and `spend`.

In [ ]:
alpha = 0.01  # Significance leel

### Checking for Randomization Balance (Baseline Characteristics)

To ensure that the observed effects on 'visit', 'conversion', and 'spend' are genuinely attributable to the e-mail campaigns and not to pre-existing differences between the groups, we need to check if the randomization successfully balanced the baseline customer characteristics across the three segments (`Mens E-Mail`, `Womens E-Mail`, `No E-Mail`).

We will compare the distributions of `recency`, `history`, `mens`, `womens`, `zip_code`, `newbie`, and `channel` across the `original_segment` groups. For categorical baseline variables, we will use Chi-squared tests, and for continuous variables, we will use ANOVA.

#### Balance Check for Categorical Baseline Variables

We will use Chi-squared tests to compare the proportions of each category for `mens`, `womens`, `newbie`, `zip_code`, `history_segment`, and `channel` across the `segment` groups.

In [ ]:
from scipy.stats import chi2_contingency


print("\n--- Balance Check for Categorical Baseline Variables ---")

campaign_related_cols = ['segment', 'visit', 'conversion', 'spend']

# Check balance for numerical columns not including campaign_related_cols
for col in numerical_cols:
    if col not in campaign_related_cols:
      print(f"\nChecking balance for '{col}':")
      contingency_table = pd.crosstab(df['segment'], df[col])
      chi2, p_value, _, _ = chi2_contingency(contingency_table)
      print(f"  Chi-squared test p-value = {p_value:.4f}")

      if p_value < alpha:
          print(f"  -> Significant imbalance detected for '{col}'. (p < {alpha})")
      else:
          print(f"  -> No significant imbalance detected for '{col}'. (p >= {alpha})")

# Check balance for categorical columns
for col in categorical_cols:
  if col not in campaign_related_cols:
    print(f"\nChecking balance for '{col}':")
    contingency_table = pd.crosstab(df['segment'], df[col])
    chi2, p_value, _, _ = chi2_contingency(contingency_table)
    print(f"  Chi-squared test p-value = {p_value:.4f}")

    if p_value < alpha:
        print(f"  -> Significant imbalance detected for '{col}'. (p < {alpha})")
    else:
        print(f"  -> No significant imbalance detected for '{col}'. (p >= {alpha})")


--- Balance Check for Categorical Baseline Variables ---

Checking balance for 'recency':
  Chi-squared test p-value = 0.9221
  -> No significant imbalance detected for 'recency'. (p >= 0.01)

Checking balance for 'history':
  Chi-squared test p-value = 0.5505
  -> No significant imbalance detected for 'history'. (p >= 0.01)

Checking balance for 'mens':
  Chi-squared test p-value = 0.6717
  -> No significant imbalance detected for 'mens'. (p >= 0.01)

Checking balance for 'womens':
  Chi-squared test p-value = 0.7289
  -> No significant imbalance detected for 'womens'. (p >= 0.01)

Checking balance for 'newbie':
  Chi-squared test p-value = 0.9339
  -> No significant imbalance detected for 'newbie'. (p >= 0.01)

Checking balance for 'history_segment':
  Chi-squared test p-value = 0.3580
  -> No significant imbalance detected for 'history_segment'. (p >= 0.01)

Checking balance for 'zip_code':
  Chi-squared test p-value = 0.5795
  -> No significant imbalance detected for 'zip_code'. (

#### Balance Check for Continuous Baseline Variables

We will use ANOVA (Analysis of Variance) to compare the means of `recency` and `history` across the `original_segment` groups.

In [ ]:
from scipy.stats import f_oneway

continuous_baseline_cols = ['recency', 'history']

print("\n--- Balance Check for Continuous Baseline Variables ---")

for col in continuous_baseline_cols:
    print(f"\nChecking balance for '{col}':")
    # Extract data for each segment
    group1 = df[df['segment'] == 'No E-Mail'][col]
    group2 = df[df['segment'] == 'Mens E-Mail'][col]
    group3 = df[df['segment'] == 'Womens E-Mail'][col]

    # Perform ANOVA test
    f_stat, p_value = f_oneway(group1, group2, group3)
    print(f"  ANOVA test p-value = {p_value:.4f}")

    if p_value < alpha:
        print(f"  -> Significant imbalance detected for '{col}'. (p < {alpha})")
    else:
        print(f"  -> No significant imbalance detected for '{col}'. (p >= {alpha})")



--- Balance Check for Continuous Baseline Variables ---

Checking balance for 'recency':
  ANOVA test p-value = 0.7631
  -> No significant imbalance detected for 'recency'. (p >= 0.01)

Checking balance for 'history':
  ANOVA test p-value = 0.6980
  -> No significant imbalance detected for 'history'. (p >= 0.01)


### Conclusion on Confounding

There is no statistically significant imbalance across the campaign groups in the baseline variables that we are not testing for a causal effect with the campaign.

As such, our subsequent tests' claims of significant imbalance in the variables observed after the campaign came out, if proven statistically significant, will give a much stronger case of a *causal* effect with the received campaign rather than a *correlation*.

### Analyzing 'Visit' Outcome (Binary)

We'll compare the proportion of customers who visited the website (within the 2 weeks following the campaign) in each group using a Chi-squared test to check for statistical significance.

In [ ]:
from scipy.stats import chi2_contingency

# Group by original_segment and calculate mean visit rate
visit_rates = df.groupby('segment')['visit'].mean().reset_index()
print("\nVisit Rates by Segment:")
print(visit_rates)

# Create contingency tables for chi-squared test
# We need to create a contingency table of 'original_segment' vs 'visit' for the relevant groups

# Mens E-Mail vs. No E-Mail (Control)
data_for_mens_comparison = df[df['segment'].isin(['Mens E-Mail', 'No E-Mail'])]
contingency_table_mens = pd.crosstab(data_for_mens_comparison['segment'], data_for_mens_comparison['visit'])
chi2_mens, p_mens, _, _ = chi2_contingency(contingency_table_mens)
print(f"\nChi-squared test (Mens E-Mail vs. No E-Mail) for Visit: p-value = {p_mens:.4f}")

# Womens E-Mail vs. No E-Mail (Control)
data_for_womens_comparison = df[df['segment'].isin(['Womens E-Mail', 'No E-Mail'])]
contingency_table_womens = pd.crosstab(data_for_womens_comparison['segment'], data_for_womens_comparison['visit'])
chi2_womens, p_womens, _, _ = chi2_contingency(contingency_table_womens)
print(f"Chi-squared test (Womens E-Mail vs. No E-Mail) for Visit: p-value = {p_womens:.4f}")

print("\nConclusion for Visit:")
if p_mens < alpha:
    print("There is a statistically significant difference in visit rates between Mens E-Mail and No E-Mail groups.")
else:
    print("There is no statistically significant difference in visit rates between Mens E-Mail and No E-Mail groups.")

if p_womens < alpha:
    print("There is a statistically significant difference in visit rates between Womens E-Mail and No E-Mail groups.")
else:
    print("There is no statistically significant difference in visit rates between Womens E-Mail and No E-Mail groups.")


Visit Rates by Segment:
         segment     visit
0    Mens E-Mail  0.182757
1      No E-Mail  0.106167
2  Womens E-Mail  0.151400

Chi-squared test (Mens E-Mail vs. No E-Mail) for Visit: p-value = 0.0000
Chi-squared test (Womens E-Mail vs. No E-Mail) for Visit: p-value = 0.0000

Conclusion for Visit:
There is a statistically significant difference in visit rates between Mens E-Mail and No E-Mail groups.
There is a statistically significant difference in visit rates between Womens E-Mail and No E-Mail groups.


### Analyzing 'Conversion' Outcome (Binary)

Similar to 'visit', we'll compare the proportion of conversions (customers who purchased something in the two weeks following the campaign) in each group using a Chi-squared test.

In [ ]:
from scipy.stats import chi2_contingency

# Group by original_segment and calculate mean conversion rate
conversion_rates = df.groupby('segment')['conversion'].mean().reset_index()
print("\nConversion Rates by Segment:")
print(conversion_rates)

# Create contingency tables for chi-squared test

# Mens E-Mail vs. No E-Mail (Control)
data_for_mens_comparison_conv = df[df['segment'].isin(['Mens E-Mail', 'No E-Mail'])]
contingency_table_mens_conv = pd.crosstab(data_for_mens_comparison_conv['segment'], data_for_mens_comparison_conv['conversion'])
chi2_mens_conv, p_mens_conv, _, _ = chi2_contingency(contingency_table_mens_conv)
print(f"\nChi-squared test (Mens E-Mail vs. No E-Mail) for Conversion: p-value = {p_mens_conv:.4f}")

# Womens E-Mail vs. No E-Mail (Control)
data_for_womens_comparison_conv = df[df['segment'].isin(['Womens E-Mail', 'No E-Mail'])]
contingency_table_womens_conv = pd.crosstab(data_for_womens_comparison_conv['segment'], data_for_womens_comparison_conv['conversion'])
chi2_womens_conv, p_womens_conv, _, _ = chi2_contingency(contingency_table_womens_conv)
print(f"Chi-squared test (Womens E-Mail vs. No E-Mail) for Conversion: p-value = {p_womens_conv:.4f}")

print("\nConclusion for Conversion:")
if p_mens_conv < alpha:
    print("There is a statistically significant difference in conversion rates between Mens E-Mail and No E-Mail groups.")
else:
    print("There is no statistically significant difference in conversion rates between Mens E-Mail and No E-Mail groups.")

if p_womens_conv < alpha:
    print("There is a statistically significant difference in conversion rates between Womens E-Mail and No E-Mail groups.")
else:
    print("There is no statistically significant difference in conversion rates between Womens E-Mail and No E-Mail groups.")


Conversion Rates by Segment:
         segment  conversion
0    Mens E-Mail    0.012531
1      No E-Mail    0.005726
2  Womens E-Mail    0.008837

Chi-squared test (Mens E-Mail vs. No E-Mail) for Conversion: p-value = 0.0000
Chi-squared test (Womens E-Mail vs. No E-Mail) for Conversion: p-value = 0.0002

Conclusion for Conversion:
There is a statistically significant difference in conversion rates between Mens E-Mail and No E-Mail groups.
There is a statistically significant difference in conversion rates between Womens E-Mail and No E-Mail groups.


### Analyzing 'Spend' Outcome (Continuous)

For the continuous 'spend' variable (amount spent by customers in the two weeks following the campaign), we'll use an independent samples t-test to compare the average spend between the treatment groups and the control group.

In [ ]:
from scipy.stats import ttest_ind

# Group by original_segment and calculate mean spend
avg_spend = df.groupby('segment')['spend'].mean().reset_index()
print("\nAverage Spend by Segment:")
print(avg_spend)

# Extract spend data for each group
control_group_spend = df[df['segment'] == 'No E-Mail']['spend']
mens_email_spend = df[df['segment'] == 'Mens E-Mail']['spend']
womens_email_spend = df[df['segment'] == 'Womens E-Mail']['spend']

# Mens E-Mail vs. No E-Mail (Control)
t_stat_mens, p_mens_spend = ttest_ind(mens_email_spend, control_group_spend)
print(f"\nIndependent t-test (Mens E-Mail vs. No E-Mail) for Spend: p-value = {p_mens_spend:.4f}")

# Womens E-Mail vs. No E-Mail (Control)
t_stat_womens, p_womens_spend = ttest_ind(womens_email_spend, control_group_spend)
print(f"Independent t-test (Womens E-Mail vs. No E-Mail) for Spend: p-value = {p_womens_spend:.4f}")

print("\nConclusion for Spend:")
if p_mens_spend < alpha:
    print("There is a statistically significant difference in average spend between Mens E-Mail and No E-Mail groups.")
else:
    print("There is no statistically significant difference in average spend between Mens E-Mail and No E-Mail groups.")

if p_womens_spend < alpha:
    print("There is a statistically significant difference in average spend between Womens E-Mail and No E-Mail groups.")
else:
    print("There is no statistically significant difference in average spend between Womens E-Mail and No E-Mail groups.")



Average Spend by Segment:
         segment     spend
0    Mens E-Mail  1.422617
1      No E-Mail  0.652789
2  Womens E-Mail  1.077202

Independent t-test (Mens E-Mail vs. No E-Mail) for Spend: p-value = 0.0000
Independent t-test (Womens E-Mail vs. No E-Mail) for Spend: p-value = 0.0011

Conclusion for Spend:
There is a statistically significant difference in average spend between Mens E-Mail and No E-Mail groups.
There is a statistically significant difference in average spend between Womens E-Mail and No E-Mail groups.


## Conclusions from A/B Testing

At the significance level of alpha=0.01, using independent t-tests, Mens E-Mail vs. No E-Mail had a statistically significant effect on post-campaign visit rates (customer visited the site within 2 weeks of the campaign), conversion (customer purchased something from the site within 2 weeks of the campaign), and spend (actual dollar amount spent in the 2 weeks following the campaign), with a p-value ≈ 0.0000 for all 3 variables. Also, Womens E-Mail versus No E-Mail had a statistically significant effect on visit rates (p-value ≈ 0.0000), conversion (p-value ≈ 0.0002), and spend (p-value ≈ 0.0011)

Based on our testing for other variables not directly related to the campaign confirming that there was no significant correlation between those variables and the campaign received, as well as the statistically significant effects of the campaign received on the post-campaign visit rates, conversion, and spend, **we can strongly infer a causal effect between the campaign received and post-campaign visit rates, conversion, and spend**